#   Tool Calling (Function Calling) + 에이전트(Agent) 개념

---

## 환경 설정 및 준비

`(1) Env 환경변수`

In [ ]:
from dotenv import load_dotenv
load_dotenv()

`(2) 기본 라이브러리`

In [ ]:
import os
from glob import glob

from pprint import pprint
import json

---

## **Tool Calling**

- **Tool Calling**은 LLM이 외부 시스템과 상호작용하기 위한 **함수 호출 메커니즘**

- LLM은 정의된 도구나 함수를 통해 **외부 시스템과 통신**하고 작업을 수행

- **Tool calling**은 모델이 시스템과 직접 상호작용할 수 있게 하는 기능

- **구조화된 출력**을 통해 API나 데이터베이스와 같은 시스템 요구사항 충족

- **스키마 기반 응답**으로 시스템 간 효율적 통신 가능


![Tool Calling Concept](https://python.langchain.com/assets/images/tool_calling_concept-552a73031228ff9144c7d59f26dedbbf.png)


[참조] https://python.langchain.com/docs/concepts/tool_calling/

---

### 1. **Tool Creation** (`@tool` 데코레이터 사용)

- **@tool 데코레이터**로 함수에 스키마 정보 추가

- **함수와 스키마** 간 자동 연결로 도구 생성

In [ ]:
from langchain_core.tools import tool
from typing import Literal

@tool
def get_weather(city: Literal["서울", "부산", "대구", "인천", "광주"]):
    """한국 주요 도시의 날씨 정보를 가져옵니다."""
    weather_data = {
        "서울": "맑음",
        "부산": "흐림",
        "대구": "맑음",
        "인천": "비",
        "광주": "구름많음"
    }
    
    if city in weather_data:
        return f"{city}은(는) {weather_data[city]}"
    else:
        raise AssertionError("지원하지 않는 도시입니다")

---
### **[실습]**

- **크로마 DB 재사용**을 위해 **프로젝트 2**의 `chromadb` 디렉토리를 현재 프로젝트 폴더에 **복사하여 활용**

- 이전 프로젝트의 **테슬라, 리비안 데이터**를 동일한 **임베딩 모델**로 검색 가능하도록 구성

- 사용자 정의 **문서 검색 도구**를 구현 (`@tool` 데코레이터 사용)

In [ ]:
# 벡터 저장소 로드 


# 검색기 지정하여 테스트 


# DB 검색하는 사용자 정의 도구 생성

---

### 2. **Tool Binding** (모델에 Tool 연결)

- **모델-도구 연결**로 입력 스키마 자동 인식

- **스키마 기반 검증**으로 올바른 입력 보장

In [ ]:
from langchain_openai import ChatOpenAI

# 모델
model = ChatOpenAI(model="gpt-4.1-nano",temperature=0)

# 도구 목록
tools = [get_weather]

# 도구를 모델에 바인딩 (bind_tools 메소드 사용)
model_with_tools = model.bind_tools([get_weather])

In [ ]:
# 사용자 쿼리를 모델에 전달
result = model_with_tools.invoke("서울 날씨 어때?")

print(result)

---
### **[실습]**

- 앞의 실습에서 정의한 도구를 llm 모델에 **바인딩** 처리
- 사용자 쿼리를 입력하여 출력 결과를 search_result 변수에 저장하고 출력 확인

In [ ]:
# 도구를 모델에 바인딩

# 도구를 사용하여 쿼리 실행


---

### 3. **Tool Calling** (모델이 Tool을 사용하는 경우)

- **스키마 기반 응답** 생성으로 정확한 입력 형식 준수

- **자동 유효성 검증**으로 오류 방지

- **구조화된 출력** 생성으로 시스템 호환성 보장

In [ ]:
# 결과 출력
for k in dict(result).keys():
    print(f"{k}: ")
    print(dict(result)[k])
    print("-"*100)

In [ ]:
# tool_calls 출력
print(result.tool_calls)

---
### **[실습]**

- search_result 변수에 저장된 tool call 내역을 확인

In [ ]:
# 결과 출력

# tool_calls 출력


---

### 4. **Tool Execution**  (Tool이 호출된 경우 실행)

- **인자 기반 실행**으로 도구 기능 수행

- **모델 제공 파라미터**로 자동화된 실행

- **실행 결과** 처리 및 반환

In [ ]:
# 함수의 인자를 직접 전달하는 방식으로 실행 -> 도구를 직접 호출
get_weather.invoke("서울")

In [ ]:
# ToolCall 객체를 전달 전달하는 방식으로 실행 -> ToolMessage 객체를 반환
get_weather.invoke(result.tool_calls[0])

---

###  Tool Calling 사용 시 **고려사항**

- **모델 호환성**이 Tool Calling 성능에 직접 영향

- **명확한 도구 정의**가 모델의 이해도와 활용도 향상

- **단순한 기능**의 도구가 더 효과적으로 작동

- **과다한 도구**는 모델 성능 저하 유발

---
### **[실습]**

- search_result 변수에 저장된 tool call 내역을 직접 도구에 적용하여 실행

In [ ]:
# ToolCall 객체를 전달 전달하는 방식으로 실행 -> ToolMessage 객체를 반환


---

## **Agent**

- **LLM(대규모 언어 모델)** 을 의사결정 엔진으로 사용하여 작업을 수행하는 시스템

- 모델은 입력된 데이터를 분석하여 **맥락에 맞는 의사결정**을 수행

- 시스템은 사용자의 요청을 이해하고 **적절한 해결책**을 제시

- 복잡한 작업을 자동화하여 **업무 효율성**을 높일 수 있음 

---

### **create_agent** 

- **create_agent**는 LangChain v1.0의 표준 에이전트 생성 함수

- LangGraph를 기반으로 구축되어 **영속성, 스트리밍, Human-in-the-loop** 등의 기능을 자동 지원

- **미들웨어**를 통한 유연한 커스터마이징 가능

`(1) 추가 도구 정의`

- **@tool 데코레이터**를 사용해 계산(파이썬 코드 실행) 기능을 가진 **커스텀 도구를 정의**

- 데코레이터를 통해 함수가 **Tool Calling 시스템에 등록**되어 LLM이 호출 가능

In [ ]:
@tool
def calculate(expression: str) -> float:
    """수학 계산을 수행합니다."""
    return eval(expression)

In [ ]:
# 도구 실행 
calculate.invoke("3+2")

`(2) create_agent로 에이전트 생성`

- **create_agent**는 모델, 도구, 시스템 프롬프트를 받아 에이전트를 생성

- **system_prompt** 매개변수로 에이전트의 역할과 행동 방식을 정의

- LangGraph 기반으로 구축되어 자동으로 메시지 기록, 도구 실행 루프 등을 관리

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

# 모델 초기화
llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)

# 도구 목록
tools = [get_weather, calculate]

# 에이전트 생성
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="당신은 사용자의 요청을 처리하는 AI Assistant입니다."
)

`(3) 에이전트 실행`

- **invoke** 메서드로 에이전트를 실행

- **messages** 키에 대화 메시지 리스트를 전달

- 에이전트는 자동으로 필요한 도구를 호출하고 최종 응답을 생성

In [ ]:
# 에이전트 실행
response = agent.invoke(
    {"messages": [{"role": "user", "content": "서울의 날씨는 어떤가요?"}]},
)

# 에이전트 실행 결과 출력
pprint(response)

In [ ]:
for msg in response['messages']:
    msg.pretty_print()

In [ ]:
# 계산 도구를 사용하는 예제
response = agent.invoke(
    {"messages": [{"role": "user", "content": "32 더하기 18은 얼마인가요?"}]},
)

# 에이전트 실행 결과 출력
for msg in response['messages']:
    msg.pretty_print()

`(4) 중간 단계 확인하기`

- **stream_mode="values"** 를 사용하여 에이전트의 실행 과정을 스트리밍으로 확인

- 각 단계에서 모델의 사고 과정과 도구 호출 내역을 추적 가능

In [ ]:
# 스트리밍 모드로 중간 단계 확인
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "32 곱하기 18은 얼마인가요?"}]},
    stream_mode="values"
):
    # 각 단계의 메시지 출력
    chunk["messages"][-1].pretty_print()

---
### **[실습]**

- 이전 [실습]에서 구현한 **문서 검색 도구**를 사용하여 에이전트 구현 및 실행 

In [ ]:
# 사용자 정의 시스템 프롬프트


# 에이전트 생성



# 에이전트 실행
